# 03 — Train GroundZero

**Run on Colab Pro (A100) or Kaggle (T4×2).** Uses mixed precision fp16 for 2–3× speedup.

 **Run on Colab T4 GPU.** Trains the full GroundZero temporal video grounding model.

## What this notebook does
1. Downloads `shaunmarvell/qvhighlights-1fps` from HuggingFace (35 GB)
2. Initialises the model (SigLIP 2 backbone shared — only loaded once)
3. Trains for 20 epochs with AdamW + cosine warmup
4. Logs to W&B every 50 steps
5. Saves best checkpoint to HuggingFace Hub

## Architecture recap
```
frames  → VisualEncoder (SigLIP 2 + LoRA) → TemporalContextModule →  ─┐
query   → TextEncoder  (SigLIP 2 shared)  →                           ├→ CrossModalTransformer → SpanHead
                                                                      ┘
```
Total trainable: ~154 M (LoRA 0.15M + TemporalConv 26.6M + CrossAttn 127.5M + SpanHead 0.67M)

## Loss
`total = span_CE + 0.5*iou + 0.1*contrastive + conf_bce`


In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
if not result.stdout:
    raise RuntimeError('No GPU found.')


In [ ]:
# ── Cell 2: Install packages ──────────────────────────────────────────────────
!pip install -q transformers==4.47.0 peft==0.14.0 huggingface_hub datasets wandb einops


In [ ]:
# ── Cell 3: Authenticate ──────────────────────────────────────────────────────
# Colab: Secrets panel (left sidebar → key icon)
# Kaggle: Settings → Secrets
try:
    from google.colab import userdata
    HF_TOKEN  = userdata.get('HF_TOKEN')
    WANDB_KEY = userdata.get('WANDB_KEY')
except ImportError:
    import os  # Kaggle: set as environment secrets
    HF_TOKEN  = os.environ.get('HF_TOKEN', '')
    WANDB_KEY = os.environ.get('WANDB_KEY', '')

from huggingface_hub import login
import wandb
login(token=HF_TOKEN)
wandb.login(key=WANDB_KEY)
print('Auth OK')


In [ ]:
# ── Cell 4: Clone repo ────────────────────────────────────────────────────────
import os, sys, shutil

REPO_DIR = '/content/groundzero' if os.path.exists('/content') else '/kaggle/working/groundzero'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone https://github.com/shaunmarv3/groundzero-backend.git {REPO_DIR}
!pip install -q -r {REPO_DIR}/requirements.txt

# GitHub repo has app/ at root (groundzero-backend/ is just a local folder name)
sys.path.insert(0, REPO_DIR)

import importlib.util
print(f'app module: {"FOUND ✓" if importlib.util.find_spec("app") else "NOT FOUND"}')


## Download Dataset (~35 GB)

Downloads ~35 GB from `shaunmarvell/qvhighlights-1fps`. First run takes ~30–60 min on Colab.
Subsequent runs in the same session reuse the cache.

> **Note:** The first 1 000 train videos are stored as individual JPEGs in the `frames/` folder
> in the HuggingFace repo (not tarred). Batches 1 000–7 445 are in tar files. All val frames
> are in `val_frames_000000_end.tar`.

In [ ]:
# ── Cell 5: Download annotations ─────────────────────────────────────────────
from huggingface_hub import hf_hub_download
from pathlib import Path
import os, json

DATASET_ID = 'shaunmarvell/qvhighlights-1fps'
DATA_DIR   = Path('/content/qvhighlights') if os.path.exists('/content') else Path('/kaggle/working/qvhighlights')
DATA_DIR.mkdir(exist_ok=True)

for fname in ['annotations_train.jsonl', 'annotations_val.jsonl']:
    hf_hub_download(DATASET_ID, fname, repo_type='dataset', local_dir=str(DATA_DIR))
    print(f'  {fname} OK')

TRAIN_JSONL = DATA_DIR / 'annotations_train.jsonl'
VAL_JSONL   = DATA_DIR / 'annotations_val.jsonl'
FRAMES_DIR  = DATA_DIR / 'frames'
FRAMES_DIR.mkdir(exist_ok=True)

n_train = sum(1 for _ in open(TRAIN_JSONL))
n_val   = sum(1 for _ in open(VAL_JSONL))
print(f'Train: {n_train}  Val: {n_val}')


In [ ]:
# ── Cell 6: Download + extract train tars ────────────────────────────────────
import tarfile

TRAIN_TARS = [
    'frames_001000_002000.tar', 'frames_002000_003000.tar',
    'frames_003000_004000.tar', 'frames_004000_005000.tar',
    'frames_005000_006000.tar', 'frames_006000_007241.tar',
    'frames_007242_007445.tar',
]
for tar_name in TRAIN_TARS:
    print(f'Downloading {tar_name}...')
    p = hf_hub_download(DATASET_ID, tar_name, repo_type='dataset', local_dir=str(DATA_DIR))
    with tarfile.open(p) as t:
        t.extractall(str(DATA_DIR))
    os.remove(p)
    print(f'  done. vids so far: {sum(1 for x in FRAMES_DIR.iterdir() if x.is_dir())}')


In [ ]:
# ── Cell 7: Download + extract val tar ───────────────────────────────────────
p = hf_hub_download(DATASET_ID, 'val_frames_000000_end.tar', repo_type='dataset', local_dir=str(DATA_DIR))
with tarfile.open(p) as t:
    t.extractall(str(DATA_DIR))
os.remove(p)
print('Val frames extracted.')


In [ ]:
# ── Cell 8: Download first batch (vids 0–1000, individual files) ──────────────
from huggingface_hub import snapshot_download
snapshot_download(
    DATASET_ID, repo_type='dataset',
    allow_patterns=['frames/**'],
    local_dir=str(DATA_DIR),
    local_dir_use_symlinks=False,
)
n_vids = sum(1 for p in FRAMES_DIR.iterdir() if p.is_dir())
print(f'Total vid dirs: {n_vids}  (expect ~8995)')


In [ ]:
# ── Cell 9: Verify data ───────────────────────────────────────────────────────
import random
from PIL import Image
with open(TRAIN_JSONL) as f:
    samples = [json.loads(l) for l in f]
s = random.choice(samples)
jpgs = sorted((FRAMES_DIR / s['vid']).glob('*.jpg'))
img  = Image.open(jpgs[0])
print(f"vid: {s['vid']}  frames: {len(jpgs)}  size: {img.size}")
assert img.size == (384, 384)
print('Data OK')


## Config


In [ ]:
# ── Cell 10: Config ───────────────────────────────────────────────────────────
# batch_size guide (after sanity check notebook confirms your actual VRAM):
#   T4  16 GB fp16 → 2
#   A100 40 GB fp16 → 8
#   H100 80 GB fp16 → 16
CFG = {
    'train_jsonl':  str(TRAIN_JSONL),
    'val_jsonl':    str(VAL_JSONL),
    'frames_dir':   str(FRAMES_DIR),
    'model_id':     'google/siglip2-so400m-patch14-384',
    'd_model':      1152,
    'n_heads':      8,
    'n_layers':     4,
    'lora_rank':    8,
    'lora_alpha':   16,
    'lora_layers':  [23, 24, 25, 26],
    'dropout':      0.1,
    'device':       'cuda',
    'batch_size':   2,       # ← set from sanity check results
    'lr':           1e-4,
    'weight_decay': 0.01,
    'epochs':       20,
    'warmup_steps': 500,
    'clip_grad':    1.0,
    'log_every':    50,
    'val_every':    1,
    'iou_weight':   0.5,
    'cont_weight':  0.1,
    'ckpt_dir':     '/content/checkpoints' if os.path.exists('/content') else '/kaggle/working/checkpoints',
    'hf_repo':      'shaunmarvell/groundzero-weights',
}
Path(CFG['ckpt_dir']).mkdir(exist_ok=True)
print('Config loaded')


## Dataset + DataLoader


In [ ]:
# ── Cell 11: Dataset + DataLoader ────────────────────────────────────────────
import torch
from torch.utils.data import DataLoader
from app.training.dataset import GroundingDataset, collate_fn

train_ds = GroundingDataset(jsonl_path=CFG['train_jsonl'], frames_root=CFG['frames_dir'], augment=True)
val_ds   = GroundingDataset(jsonl_path=CFG['val_jsonl'],   frames_root=CFG['frames_dir'], augment=False)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True, persistent_workers=True)

print(f'Train: {len(train_ds)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} samples,   {len(val_loader)} batches')


## Model


In [ ]:
# ── Cell 12: Build model ──────────────────────────────────────────────────────
# Kaggle T4×2 note: our architecture encodes frames BEFORE model.forward(),
# so DataParallel doesn't help the real bottleneck (SigLIP frame encoding).
# Both T4s are available but we use GPU 0 only. This is fine for first training run.
# Only 1 T4 = 16 GB VRAM. Sanity check notebook tells exact safe batch_size.
from app.pipeline.groundzero_model import GroundZeroModel

model = GroundZeroModel(
    model_id=CFG['model_id'], d_model=CFG['d_model'], n_heads=CFG['n_heads'],
    n_layers=CFG['n_layers'], lora_rank=CFG['lora_rank'], lora_alpha=CFG['lora_alpha'],
    lora_layers=CFG['lora_layers'], dropout=CFG['dropout'], device=CFG['device'],
)
counts = model.count_params()
print(f"Trainable: {counts['total_trainable']/1e6:.1f} M  Frozen: {counts['total_frozen']/1e6:.1f} M")
print(f"GPUs available: {torch.cuda.device_count()} — using GPU 0")


In [ ]:
# ── Cell 13: Optimizer + Scheduler + GradScaler ───────────────────────────────
from transformers import get_cosine_schedule_with_warmup

decay_params, no_decay_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    (no_decay_params if ('lora_' in name or 'bias' in name) else decay_params).append(param)

optimizer = torch.optim.AdamW(
    [{'params': decay_params, 'weight_decay': CFG['weight_decay']},
     {'params': no_decay_params, 'weight_decay': 0.0}],
    lr=CFG['lr'],
)
total_steps = len(train_loader) * CFG['epochs']
scheduler = get_cosine_schedule_with_warmup(optimizer, CFG['warmup_steps'], total_steps)

# GradScaler: prevents fp16 underflow during backward pass.
# Scales loss up before backward, scales gradients back down before optimizer step.
scaler = torch.cuda.amp.GradScaler()

print(f'AdamW lr={CFG["lr"]}  decay={len(decay_params)} params  no-decay={len(no_decay_params)} params')
print(f'Cosine scheduler  warmup={CFG["warmup_steps"]}  total={total_steps} steps')
print('GradScaler ready')


## Training Step (mixed precision)

`autocast` runs the forward pass in fp16 — faster on TensorCores, half the activation memory.  
`GradScaler` handles the backward pass safely so fp16 gradients don't underflow to zero.


In [ ]:
# ── Cell 14: Encode batch helper ─────────────────────────────────────────────
import torch.nn.functional as F
DEVICE = CFG['device']

def encode_batch(batch):
    """
    Encodes frames + queries. Called inside autocast context from train_step/validate,
    so all model ops here automatically run in fp16.

    Returns:
        frame_embs : (B, N_max, 1152)
        timestamps : (B, N_max)  fractional t/duration, padded with 1.0
        query_emb  : (B, 1, 1152)
    """
    B, D = len(batch['frames']), CFG['d_model']
    emb_list, ts_list = [], []

    for i in range(B):
        emb_i  = model.visual_encoder.encode_frames(batch['frames'][i])   # (N_i, D)
        frac_i = torch.tensor(
            [t / batch['duration'][i].item() for t in batch['timestamps'][i]],
            dtype=torch.float32, device=DEVICE
        )
        emb_list.append(emb_i)
        ts_list.append(frac_i)

    N_max      = max(e.shape[0] for e in emb_list)
    frame_embs = torch.zeros(B, N_max, D, device=DEVICE)
    timestamps = torch.ones(B, N_max, device=DEVICE)  # 1.0 pad = end-of-video position
    for i, (emb, ts) in enumerate(zip(emb_list, ts_list)):
        n = emb.shape[0]
        frame_embs[i, :n] = emb
        timestamps[i, :n] = ts

    query_emb = model.text_encoder.encode_queries(batch['query']).unsqueeze(1)  # (B, 1, D)
    return frame_embs, timestamps, query_emb


In [ ]:
# ── Cell 15: Training step ────────────────────────────────────────────────────
from app.training.losses import combined_loss

def train_step(batch):
    model.train()
    optimizer.zero_grad()

    # autocast: forward pass runs in fp16 — faster + less memory
    with torch.cuda.amp.autocast():
        frame_embs, timestamps, query_emb = encode_batch(batch)
        start_logits, end_logits, confidence = model(frame_embs, timestamps, query_emb)

        gt_start = batch['gt_start_idx'].to(DEVICE)
        gt_end   = batch['gt_end_idx'].to(DEVICE)
        is_neg   = batch['is_negative'].to(DEVICE)

        pos_mask = ~is_neg
        if pos_mask.any():
            total, span_l, iou_l, cont_l = combined_loss(
                start_logits[pos_mask], end_logits[pos_mask],
                gt_start[pos_mask], gt_end[pos_mask],
                query_emb[pos_mask], frame_embs[pos_mask],
            )
        else:
            total = span_l = iou_l = cont_l = torch.tensor(0.0, device=DEVICE)

        conf_targets = torch.where(is_neg, torch.zeros_like(confidence), torch.ones_like(confidence))
        conf_l = F.binary_cross_entropy(confidence, conf_targets)
        loss   = total + conf_l

    # scaler.scale: multiplies loss by scale factor so fp16 gradients don't underflow
    scaler.scale(loss).backward()
    # unscale_ before clipping so clip threshold is in the correct scale
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], CFG['clip_grad']
    )
    scaler.step(optimizer)   # unscales gradients then calls optimizer.step()
    scaler.update()          # adjusts scale factor for next iteration
    scheduler.step()

    return {
        'train/loss':      loss.item(),
        'train/span_loss': span_l.item(),
        'train/iou_loss':  iou_l.item(),
        'train/cont_loss': cont_l.item(),
        'train/conf_loss': conf_l.item(),
        'train/lr':        scheduler.get_last_lr()[0],
    }


## Validation

**R@1 IoU=0.5** — fraction of queries where predicted span overlaps GT by ≥ 50%.


In [ ]:
# ── Cell 16: Validation ───────────────────────────────────────────────────────
from app.pipeline.span_extraction import decode_best_span, to_seconds

def temporal_iou(s1, e1, s2, e2):
    inter = max(0.0, min(e1, e2) - max(s1, s2))
    union = max(e1, e2) - min(s1, s2)
    return inter / union if union > 0 else 0.0

@torch.no_grad()
def validate():
    model.eval()
    hits_05 = hits_07 = total = 0
    val_loss = 0.0

    for batch in val_loader:
        # autocast in validation too — same speed benefit, no scaler needed (no backward)
        with torch.cuda.amp.autocast():
            frame_embs, timestamps, query_emb = encode_batch(batch)
            start_logits, end_logits, confidence = model(frame_embs, timestamps, query_emb)

        for i in range(start_logits.shape[0]):
            dur_i    = batch['duration'][i].item()
            n_frames = len(batch['frames'][i])
            s_idx, e_idx = decode_best_span(start_logits[i], end_logits[i])
            s_idx = min(s_idx, n_frames - 1)
            e_idx = min(e_idx, n_frames - 1)
            pred_s, pred_e = to_seconds(s_idx, e_idx, n_frames, dur_i)

            ts_i = batch['timestamps'][i]
            gt_s = ts_i[min(batch['gt_start_idx'][i].item(), len(ts_i)-1)]
            gt_e = ts_i[min(batch['gt_end_idx'][i].item(),   len(ts_i)-1)]

            iou = temporal_iou(pred_s, pred_e, gt_s, gt_e)
            hits_05 += iou >= 0.5
            hits_07 += iou >= 0.7
            total   += 1

        pos = ~batch['is_negative'].to(DEVICE)
        if pos.any():
            with torch.cuda.amp.autocast():
                loss, *_ = combined_loss(
                    start_logits[pos], end_logits[pos],
                    batch['gt_start_idx'].to(DEVICE)[pos],
                    batch['gt_end_idx'].to(DEVICE)[pos],
                    query_emb[pos], frame_embs[pos],
                )
            val_loss += loss.item()

    return {
        'val/r1_iou05': hits_05 / total,
        'val/r1_iou07': hits_07 / total,
        'val/loss':     val_loss / len(val_loader),
    }


## Training Loop


In [ ]:
# ── Cell 17: Training loop ────────────────────────────────────────────────────
import wandb

run = wandb.init(project='groundzero', name=f'train-siglip2-lora-r{CFG["lora_rank"]}', config=CFG)

best_r1_05  = 0.0
global_step = 0
ckpt_dir    = Path(CFG['ckpt_dir'])

for epoch in range(1, CFG['epochs'] + 1):
    epoch_loss, n_batches = 0.0, 0

    for batch in train_loader:
        m = train_step(batch)
        epoch_loss += m['train/loss']
        n_batches  += 1
        global_step += 1

        if global_step % CFG['log_every'] == 0:
            wandb.log(m, step=global_step)
            print(f'ep {epoch:02d}  step {global_step:05d}  '
                  f'loss={m["train/loss"]:.4f}  span={m["train/span_loss"]:.4f}  '
                  f'iou={m["train/iou_loss"]:.4f}  lr={m["train/lr"]:.2e}')

    print(f'--- Epoch {epoch:02d} avg loss: {epoch_loss/n_batches:.4f} ---')

    if epoch % CFG['val_every'] == 0:
        vm = validate()
        wandb.log({**vm, 'epoch': epoch}, step=global_step)
        print(f'  Val R@1 IoU=0.5: {vm["val/r1_iou05"]:.4f}  IoU=0.7: {vm["val/r1_iou07"]:.4f}')

        if vm['val/r1_iou05'] > best_r1_05:
            best_r1_05 = vm['val/r1_iou05']
            ckpt_path  = ckpt_dir / f'best_ep{epoch:02d}_r105_{best_r1_05:.4f}.pt'
            torch.save({
                'epoch': epoch, 'step': global_step,
                'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                'r1_iou05': best_r1_05, 'cfg': CFG,
            }, ckpt_path)
            print(f'  Saved → {ckpt_path.name}')
            wandb.save(str(ckpt_path))

run.finish()
print(f'Done. Best R@1 IoU=0.5: {best_r1_05:.4f}')


## Push to HuggingFace


In [ ]:
# ── Cell 18: Push best checkpoint ─────────────────────────────────────────────
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(CFG['hf_repo'], repo_type='model', exist_ok=True)
best_ckpt = max(ckpt_dir.glob('best_*.pt'), key=lambda p: p.stat().st_mtime)
api.upload_file(path_or_fileobj=str(best_ckpt), path_in_repo=best_ckpt.name,
                repo_id=CFG['hf_repo'], repo_type='model')
print(f'Uploaded {best_ckpt.name}')


## Resume from Checkpoint


In [ ]:
# ── Cell 19: Resume (uncomment to use) ───────────────────────────────────────
# RESUME = '/content/checkpoints/best_ep05_r105_0.3210.pt'
# ckpt = torch.load(RESUME, map_location=DEVICE)
# model.load_state_dict(ckpt['model'])
# optimizer.load_state_dict(ckpt['optimizer'])
# scheduler.load_state_dict(ckpt['scheduler'])
# scaler.load_state_dict(ckpt['scaler'])
# start_epoch = ckpt['epoch'] + 1
# global_step = ckpt['step']
# best_r1_05  = ckpt['r1_iou05']
# print(f'Resumed from epoch {ckpt["epoch"]}, best R@1={best_r1_05:.4f}')
